# Get to Know a Dataset: LAION-fMRI

This notebook serves as a guided tour of the [LAION-fMRI](https://registry.opendata.aws/laion-fmri) dataset: one of the largest high-resolution 7T fMRI datasets of human brain responses to natural images. Five participants viewed ~25,000 in total across 34 scanning sessions per participant, at 1.8 mm isotropic resolution. The dataset also includes eye-tracking, behavior, retinotopy, functional localizers, and precision diffusion MRI.

More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/). Full project documentation lives at [laion-fmri.hebartlab.com](https://laion-fmri.hebartlab.com).

> **DRAFT NOTEBOOK** — this is a stub. Sections are sketched out but most code cells are placeholders. Content will be filled in as the dataset documentation and S3 release are finalized.

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of the dataset.
1. How are key prefixes and objects organized in the S3 bucket?
2. What kinds of filetypes are represented?
3. Explain with text and demonstrate with code.

DATA PROVIDER INSTRUCTIONS -->

## Q: How is the dataset organized? Help us understand the key prefix structure of the S3 bucket.

LAION-fMRI follows the [Brain Imaging Data Structure (BIDS)](https://bids.neuroimaging.io/) specification, which is the community standard for organizing neuroimaging datasets. At the top level of the bucket you will find:

1. `dataset_description.json`, `participants.tsv`, `README`, and `CHANGES` — BIDS metadata
2. `sub-01/` ... `sub-05/` — one prefix per participant, each containing per-session functional, anatomical, eye-tracking, and diffusion data
3. `derivatives/` — preprocessed data, GLMsingle single-trial betas, retinotopy maps, functional localizer contrasts, ROI masks, and diffusion tractography
4. `stimuli/` — the natural image stimuli (and pointers to the source LAION images)

Next, define the bucket location and create an unsigned `boto3` S3 client (the bucket is public, so no AWS credentials are required).

In [ ]:
# TODO: replace with the published bucket name once available
bucket = "[BUCKET_NAME]"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# List the top-level prefixes
# response = s3.list_objects_v2(Bucket=bucket, Delimiter="/")
# for item in response.get("CommonPrefixes", []):
#     print(item["Prefix"])

In [ ]:
# Look inside one participant prefix to see the per-session BIDS layout
# for item in s3.list_objects_v2(Bucket=bucket, Prefix="sub-01/", MaxKeys=20)["Contents"]:
#     print(item["Key"])

<!-- DATA PROVIDER INSTRUCTIONS
Explain data formats present, why they were chosen, tooling recommendations,
and any dataset-specific aspects.
DATA PROVIDER INSTRUCTIONS -->

## Q: What data formats are in the dataset, and how should I work with them?

LAION-fMRI uses standard neuroimaging and tabular formats organized under BIDS:

- **NIfTI (`.nii.gz`)** — 4D functional volumes, 3D anatomical volumes, masks, statistical maps. The de facto standard for MRI data. Read with [`nibabel`](https://nipy.org/nibabel/) or [`nilearn`](https://nilearn.github.io/).
- **TSV / JSON sidecars** — BIDS metadata, event timing, participant tables. Read with `pandas` and the built-in `json` library.

... and more

> **TODO**: add more details on the exact formats and tooling recommendations.

<!-- DATA PROVIDER INSTRUCTIONS
Demonstrate loading a portion of data from the dataset and reveal something
about its structure.
DATA PROVIDER INSTRUCTIONS -->

## Q: Can you show us an example of downloading and loading data from the dataset?
To make use of our dataset as easy as possible, we provide a Python package that allows users to:
- Download the data (or subsets) from the AWS S3 bucket
- Load specific parts of the data into memory (e.g. single-trial beta maps for a given participant, run, and stimulus)
- Load supplementary data (e.g. retinotopy maps, stimuli, behavioural data, ROIs)
- Easily transform data between different brain spaces (e.g. from MNI to native space or vice versa)

**Note:** You can find the full package documentation at [laion-fmri.hebartlab.com/data_access](https://laion-fmri.hebartlab.com/data_access).

To get started, install the package using pip:

```bash
pip install laion-fmri
```
 
then follow the examples below.

In [ ]:
# TODO: data loader examples

<!-- DATA PROVIDER INSTRUCTIONS
Visualize some aspect of the dataset to help users understand it. Goal: inform
AND impress.
DATA PROVIDER INSTRUCTIONS -->

## Q: A picture is worth a thousand words. Show us a visual from the dataset.

Some visualizations we plan to include in the final notebook:

1. **A summary plot** of stimuli per session per participant, to give a sense of dataset scale
2. **Mosaic images** of the high-resolution T1w anatomical for one participant
3. **A functional activation map** for the face / scene / body / word functional localizers and **example BOLD time series** for selected high reliability voxels
4. **An example single-trial beta map** for one stimulus image, side-by-side with the image itself and **surface maps of voxel reliability measures**

In [ ]:
# TODO: implement visualizations

<!-- DATA PROVIDER INSTRUCTIONS
Show an opinionated example of answering a question with the dataset. A toy
example is fine. Should transmit domain experience and serve as a jumping-off
point for users.
DATA PROVIDER INSTRUCTIONS -->

## Q: What is one question you have answered with these data? Show us how.

One of the main questions we were interested in when creating this dataset was whether the recent finding that all vision models converge to a similar performance, independent of training data, architecture or task (Conwell et al., 2024), would replicate in a more diverse dataset. 

We found that the answer is yes! To do so, we fully replicated the analysis from Conwell et al., 2024, using the same model architectures, training procedures, and evaluation metrics, but trained on our dataset. The following code shows a minimal example of how this works. 



**Step 1 — Load betas and stimuli for one session of one participant.** We restrict ourselves to a single session here so the example runs quickly on a laptop; the same code generalizes to the full dataset by running it on concatenated data from all sessions.

We use the convenience loader from the `laion-fmri` package to fetch the GLMsingle single-trial beta maps (already in the participant's native functional space, masked to a visual cortex ROI for this example) along with the stimulus images that were shown on each trial.

In [ ]:
# TODO: replace with the real loader API once finalized
# from laion_fmri import load_betas, load_stimuli
#
# subject = "sub-01"
# session = "ses-01"
# roi     = "visual_cortex"  # union of early visual + ventral / lateral / dorsal streams
#
# # betas: (n_trials, n_voxels)  -- single-trial GLMsingle estimates
# # stim_ids: (n_trials,)        -- index into the stimulus image array
# betas, stim_ids = load_betas(subject=subject, session=session, roi=roi, space="native")
#
# # images: (n_unique_stimuli, H, W, 3) uint8
# images = load_stimuli(stim_ids)
#
# print("betas:", betas.shape, "stimuli:", images.shape)

**Step 2 — Extract image features from a vision model.** For this minimal example we use AlexNet pretrained on ImageNet and pull activations from a single intermediate layer (`features.7`, the third conv block after ReLU). Any `torchvision` model and any layer would work; the point is to get one feature vector per image.

In [ ]:
import torch
import torchvision.models as tvm
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
model = tvm.alexnet(weights=tvm.AlexNet_Weights.IMAGENET1K_V1).to(device).eval()

preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Tap activations from a single intermediate layer
layer = model.features[7]  # arbitrary mid-level conv block
_activations = {}
def _hook(_, __, out): _activations["x"] = out.detach().flatten(1).cpu()
layer.register_forward_hook(_hook)

feats = []
with torch.no_grad():
    for img in images:                       # placeholder loop; batch in real code
        x = preprocess(img).unsqueeze(0).to(device)
        _ = model(x)
        feats.append(_activations["x"])
feats = torch.cat(feats, dim=0).numpy()       # (n_trials, n_features)
print("features:", feats.shape)

**Step 3 — Train/test split.** We use a simple odd/even split over trials: even-indexed trials go to train, odd to test. This directly replicates the train/test split used by Conwell et al., 2024. - other splits could potentially reveal interesting differences in results, but for the sake of replicating the original method perfectly, we use this split.

In [ ]:
n_trials = betas.shape[0]
train_idx = np.arange(0, n_trials, 2)
test_idx  = np.arange(1, n_trials, 2)

X_train, X_test = feats[train_idx], feats[test_idx]
Y_train, Y_test = betas[train_idx], betas[test_idx]

**Step 4 — Fit a voxel-wise Ridge encoding model.** For each voxel, we learn a linear mapping from image features to BOLD response. We use `RidgeCV` with `alpha_per_target=True`, which runs efficient leave-one-out CV across an alpha grid and picks the best regularization strength independently for each voxel. This matters because optimal alphas can vary substantially across visual cortex.

In [ ]:
from sklearn.linear_model import RidgeCV

# Per-voxel alpha selection: RidgeCV with `alpha_per_target=True` runs efficient
# leave-one-out CV across the alpha grid and picks the best alpha independently
# for each voxel. Optimal regularization varies a lot across visual cortex
# (early visual voxels with high SNR want smaller alphas than higher-level
# voxels with noisier responses).
alphas = np.logspace(0, 6, 13)
encoder = RidgeCV(alphas=alphas, alpha_per_target=True)
encoder.fit(X_train, Y_train)

Y_pred = encoder.predict(X_test)   # (n_test_trials, n_voxels)
print("median selected alpha:", float(np.median(encoder.alpha_)))

**Step 5 — Evaluate with voxel-encoded RSA.** Instead of asking "how well does the model predict each voxel?" we ask "does the *geometry* of model-predicted responses match the geometry of measured responses across stimuli?" Concretely:

1. Build a representational dissimilarity matrix (RDM) over test trials from the **measured** betas, using correlation distance.
2. Build the matching RDM from the **predicted** betas.
3. Score the encoding model by the Spearman correlation between the upper triangles of the two RDMs.

In [ ]:
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

rdm_measured  = pdist(Y_test, metric="correlation")
rdm_predicted = pdist(Y_pred, metric="correlation")

rsa_score, _ = spearmanr(rdm_measured, rdm_predicted)
print(f"voxel-encoded RSA score (Spearman r): {rsa_score:.3f}")

**Step 6 — Visualize the result.** A simple way to inspect the RSA evaluation is to plot the two RDMs side by side, plus a scatter of measured vs. predicted dissimilarities.

In [ ]:
from scipy.spatial.distance import squareform

fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=100)

axes[0].imshow(squareform(rdm_measured), cmap="viridis")
axes[0].set_title("Measured RDM")
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].imshow(squareform(rdm_predicted), cmap="viridis")
axes[1].set_title("Predicted RDM (AlexNet features.7)")
axes[1].set_xticks([]); axes[1].set_yticks([])

axes[2].scatter(rdm_measured, rdm_predicted, s=4, alpha=0.3, color="#4c6ef5")
axes[2].set_xlabel("Measured dissimilarity")
axes[2].set_ylabel("Predicted dissimilarity")
axes[2].set_title(f"Spearman r = {rsa_score:.3f}")
axes[2].spines["top"].set_visible(False)
axes[2].spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Visualization of full replication

When the above analysis is run on the full dataset and across many more models, we found that the results almost perfectly replicate the findings from Conwell et al., 2024.:

In [ ]:
# TODO: add plot of Conwell replication

<!-- DATA PROVIDER INSTRUCTIONS
Pose a community challenge: an unanswered question or wishlist item that's
realistic for someone to attempt with the published data.
DATA PROVIDER INSTRUCTIONS -->

## Q: What is one unanswered question that could be tackled with these data?

Many of the recent advances and findings in NeuroAI were based on datasets that less diverse than LAION-fMRI. 

We are curious to see if they will replicate in our dataset. To facilitate this, we are releasing the dataset in conjunction with the [re:vision replication challenge](https://re-vision-initiative.org), which we invite the scientific community to participate in.

---

**Reminder for the data provider**: clear all cell outputs before committing this notebook to GitHub.